# Portfolio API — INDmoney (`slug: indmoney`)

Exercises all `/portfolio/*` endpoints scoped to the `indmoney` source.
US equity holdings via CDP browser fetch — attaches to the authenticated
INDmoney web app and intercepts the holdings XHR.

**Auth pre-req:** Log in to [indmoney.com](https://indmoney.com) inside the AlphaForge Chrome session (`--remote-debugging-port=9299`). Set `INDMONEY_USER_ID` in `backend/.env.cred.local`.

In [ ]:
import json, os
from pathlib import Path

SLUG = "indmoney"
MODE = "http"          # "in_process" | "http"
BASE = "http://localhost:8000/api/v1"

AF_USERNAME = os.getenv("AF_USERNAME", "admin")
AF_PASSWORD = os.getenv("AF_PASSWORD", "alphaforge-dev")

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=60.0)
    PREFIX = ""


def _login() -> str:
    r = client.post(
        f"{PREFIX}/auth/token",
        data={"username": AF_USERNAME, "password": AF_PASSWORD},
    )
    if r.status_code != 200:
        raise RuntimeError(
            f"Auth failed ({r.status_code}): {r.text}. "
            "Set AF_USERNAME / AF_PASSWORD env vars if you changed admin creds."
        )
    return r.json()["access_token"]


def _ensure_auth() -> None:
    if "Authorization" not in client.headers:
        client.headers["Authorization"] = f"Bearer {_login()}"


def _request(method: str, path: str, **kw):
    _ensure_auth()
    r = client.request(method, f"{PREFIX}{path}", **kw)
    if r.status_code == 401:
        client.headers["Authorization"] = f"Bearer {_login()}"
        r = client.request(method, f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text


def get(path, **kw):  return _request("GET", path, **kw)
def post(path, **kw): return _request("POST", path, **kw)

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

_ensure_auth()
print(f"Mode: {MODE}  slug: {SLUG}  authed as: {AF_USERNAME}")

## 1. Source info

`status: ready` when `INDMONEY_USER_ID` is set in `.env.cred.local`, `unconfigured` otherwise.

In [ ]:
status, body = get(f"/portfolio/sources/{SLUG}")
print(status)
pp(body)

## 2. Sync

Attaches to Chrome via CDP, navigates to `indmoney.com/dashboard`, and intercepts
the US-stocks holdings XHR. Result is cached to
`~/.alphaforge/portfolio-dumps/indmoney-holdings-live.csv`.

> Requires `MODE="http"` with a live server and an open Chrome session
> where you are already logged in to indmoney.com.

In [ ]:
status, body = post(f"/portfolio/sources/{SLUG}/sync")
print(status, f"  holdings={body.get('holdings_count')}  status={body.get('info', {}).get('status')}")
for h in (body.get("holdings") or [])[:5]:
    print(f"  {h['symbol']:14}  qty={h['quantity']:<6}  avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 3. Holdings — indmoney only

US equity holdings (INDmoney's US stocks portfolio).

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print(status, "  totals:", body.get("totals"))
print(f"\n{len(body.get('holdings', []))} holdings:")
for h in body.get("holdings", []):
    print(f"  {h['symbol']:14}  qty={h['quantity']:<6}  avg=₹{h['avg_price']:>10,.2f}  ltp=₹{h['last_price']:>10,.2f}  pnl={h['pnl_pct']:>+.1f}%")

## 4. Allocation (indmoney)

Expected: `equity`-only (US stocks) allocation.

In [ ]:
status, body = get("/portfolio/holdings", params={"source": SLUG})
print("Allocation:")
for a in body.get("allocation", []):
    print(f"  {a['asset_class']:12} ₹{a['value']:>14,.0f}  ({a['pct']:>5.1f}%)")

## 5. Treemap (indmoney)

In [ ]:
status, body = get("/portfolio/treemap", params={"source": SLUG})
print(status)
for c in (body.get("cells") or [])[:10]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 6. Rebalance (indmoney)

In [ ]:
status, body = get("/portfolio/rebalance", params={"source": SLUG})
print("Drift:")
for d in body.get("drift", []):
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body.get("suggestions", []):
    print("  -", s["action"])

## 7. Standalone dump (bypass FastAPI)

Directly runs the CDP fetch + CSV write without starting the server.
Useful for testing auth and CSV output end-to-end.

In [ ]:
import asyncio, sys
sys.path.insert(0, str(Path.cwd().parent))  # add backend/ to path

from app.modules.brokers.indmoney.indmoney_dump import dump_indmoney

path = await dump_indmoney()
print(f"Dumped → {path}")

## 8. Reset indmoney cache

In [ ]:
if MODE == "in_process":
    from app.modules.brokers.registry import SOURCES
    SOURCES[SLUG].reset()
    status, body = get(f"/portfolio/sources/{SLUG}")
    print(f"{SLUG}: status={body['status']}  holdings={body['holdings_count']}")
else:
    print("Switch MODE to 'in_process' to reset the in-memory cache directly.")
    print("Or restart the server to clear all cached holdings.")